### Install required packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install openai --quiet

### Import required libraries

In [2]:
from openai import OpenAI
import csv
import time
import pandas as pd
from datetime import datetime
import os
from google.colab import userdata
import json

###Set the API KEY using Google colab user data option

In [3]:
openai_access = userdata.get('open_ai')
os.environ['OPENAI_API_KEY'] = openai_access


### Read the configurations

In [ ]:
json_file_path = "/content/drive/MyDrive/Colab Notebooks/cypress-final-code/configure.json"
with open(json_file_path, "r") as file:
    configuration = json.load(file)

### Load the BDD scenarios to data frame

In [7]:
cypress_input_csv_path = configuration['cypress_input_csv_path']
# Read the complete CSV file as a pandas DataFrame
cypress_data_frame = pd.read_csv(cypress_input_csv_path)

# GPT4 Zero-Shot Cypress code generation


In [ ]:
# read model, configuration parameters
#step 1
gpt_3_model=configuration['gpt_3.5_model']
gpt_4_model=configuration['gpt_4_model'] #gpt-4-turbo-2024-04-09
gpt_temperature=configuration['gpt_temperature']
gpt_top_p=configuration['gpt_top_p']
gpt_max_tokens=configuration['gpt_max_tokens']
#step 3
system_prompt=configuration['system_prompt_code']
openaiclient = OpenAI()
# Iterate over each row to read BDD sceanrios/Feature File
for index, row in cypress_data_frame.iterrows(): #cypress_data_frame.head(3).iterrows()
    bdd = row['Feature']
    print(bdd)
    #step 3
    zero_shot_array=configuration['zero_shot_code']
    zero_shot=' '.join(zero_shot_array)
    user_message =   zero_shot+bdd
    #call the openai api for generating feature file by proving zero or few shot prompts
    #step 4
    chat_suggestion = openaiclient.chat.completions.create(
    messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_message},
            ],
            #model=gpt_4_model,
            model=gpt_4_model,
            temperature=gpt_temperature,
            frequency_penalty=0,
            presence_penalty=0,
            top_p=gpt_top_p,
           # max_tokens=gpt_max_tokens,

        )
    with open('/content/drive/MyDrive/Colab Notebooks/cypress-final-code/gpt-4o-2024-05-13-zero-shotlog.txt', 'a') as log_file:
    # Write output to log.txt
      log_file.write(str(chat_suggestion) + '\n')
    # get the response - Cypress implementation for the given bdd scenarios
    output_response = chat_suggestion.choices[0].message.content
    print(output_response)
    cypress_data_frame.at[index, 'gpt-4o-2024-05-13'] = output_response
    time.sleep(2)
cypress_data_frame.to_csv(cypress_input_csv_path, index=False)

## Cypress Cucumber Implementation by following few-shot chain prompt technique.

In [ ]:
# read model, configuration parameters
#step 2
#gpt_3_model=configuration['gpt_3.5_model']
gpt_4_model=configuration['gpt_4_model']
gpt_temperature=configuration['gpt_temperature']
gpt_top_p=configuration['gpt_top_p']
gpt_max_tokens=configuration['gpt_max_tokens']
#step 3
system_prompt=configuration['system_prompt_code']
openaiclient = OpenAI()
few_shot_assistant_feature_code = ' '.join(configuration['few_shot_assistant_feature_code'])
few_shot_assistant_step_code = ' '.join(configuration['few_shot_assistant_step_code'])
few_shot_assistant_page_code = ' '.join(configuration['few_shot_assistant_page_code'])
# Iterate over each row to generate step definition implementation for a given feature by giving example code following Few-shot technique.
for index, row in cypress_data_frame.iterrows(): #cypress_data_frame.head(3).iterrows():
    bdd = row['Feature']
    print(bdd)
    #step 3
    chat_suggestion = openaiclient.chat.completions.create(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "for {feature} ="+few_shot_assistant_feature_code},
            {"role": "assistant", "content": "the example step definition javascript file is {step} = "+few_shot_assistant_step_code},
            {"role": "user", "content": "for {feature} = " + str(bdd) + "lets think step by step."+
             "Step 1: Strictly follow exact syntax for importing page object class.follow lower came case name convention, The example syntax is - import {loginPage} from '@pages/LoginPage'"+
             "Step 2: When importing more than one Page Object Class, write each import separately. For example, the following is incorrect syntax:"+
              "import { calcPage, compPage } from '@pages'; "+
              "The correct syntax is: "+
              "import { calcPage } from '@pages/CalcPage'; "+
              "import { compPage } from '@pages/CompPage'; "+
             "Write the step definitions for every scenario and the javascript file is {step} = "

             },
        ],




        model=gpt_4_model,
        #max_tokens=4050,
        top_p=1,
        temperature=0,
        )
    time.sleep(10)
    step=chat_suggestion.choices[0].message.content
    with open('/content/drive/MyDrive/Colab Notebooks/cypress-final-code/gpt-4o-2024-08-06-fewshot-log.txt', 'a') as log_file:
    # Write output to log.txt
      log_file.write(str(chat_suggestion) + '\n')
    # Iterate over each generated step definition implementation to generate POM class code by giving example code following Few-shot technique.
    chat_suggestion = openaiclient.chat.completions.create(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": "for the {step} file =" + few_shot_assistant_step_code},
            {"role": "assistant", "content":  "the page object class  implementation for {page} = " + few_shot_assistant_page_code},
             {"role": "user", "content": "lets think step by step."+
              //"Step 1: Implement the Page Object Class for every page imported in the step definition file." +
              "Step 1: Define all Web element methods along with their selectors." +
              "Step 2: Write the complete Web element interaction logic in the UI methods."+
              "Step 3: Provide a complete implementation for every verification method."+
              "Step 4: In the absence of implementation details, assume and write the code. Do not leave any method without full implementation."+
              "Step 5: If application behavior or functionality is unclear, assume the most common behavior/functionality."+
              "Step 6: Avoid leaving any placeholders; implement the full logic for each method."+
              "Step 7: Ensure that the Page Object Class is implemented for all page imports in the step definition file."+
              "Step 8: Ensure the Cypress code does not have any JavaScript type errors."+
              "Step 9: Review the generated code thoroughly. Verify each step, and if any step is not followed, rewrite the code accordingly." +
                         "for the given {step} file =" +str(step) +"the page object class implementation is {page} = "
            },
        ],
        model=gpt_4_model,
        top_p=1,
        temperature=0,
        )
    page=chat_suggestion.choices[0].message.content
    with open('/content/drive/MyDrive/Colab Notebooks/cypress/gpt-4o-2024-08-06-fewshot-log.txt', 'a') as log_file:
    # Write output to log.txt
      log_file.write(str(chat_suggestion) + '\n')
    test_code=step+page
    print(test_code)
    cypress_data_frame.at[index, 'gpt-4o-2024-08-06-fewshot'] = test_code
    print(index)
    time.sleep(10)
cypress_data_frame.to_csv(cypress_input_csv_path, index=False)